## CS152 Final Project

**Sources:** 

https://www.youtube.com/watch?v=l-hh51ncgDI


## Set Up Board & Move Evaluation

### Load Imports & Data

In [ ]:
import numpy as np
import random
import sys
import math
import copy
import re
import time
import os
import json
from dotenv import load_dotenv
import matplotlib.pyplot as plt
from openai import OpenAI
from anthropic import Anthropic
load_dotenv()

True

In [ ]:
# Load generated positions
with open('generated_positions.json', 'r') as f:
    positions_data = json.load(f)

p1_win_in_1 = positions_data["p1_win_in_1"]
p2_win_in_1_block = positions_data["p2_win_in_1_block"]
p1_win_in_3 = positions_data["p1_win_in_3"]
p2_win_in_3 = positions_data["p2_win_in_3"]
p1_win_in_5 = positions_data["p1_win_in_5"]
p2_win_in_5 = positions_data["p2_win_in_5"]
early_after_5_moves = positions_data["early_after_5_moves"]
late_after_20_moves = positions_data["late_after_20_moves"]

### Board & Helpers

In [ ]:
def create_new_board(columns, rows): 
    """
    Initialize an empty board with a given number of columns and rows. 

    Input: 
        columns: the number of columns in the board
        rows: the number of rows in the board

    Output:
        the board as a list of lists
    """
    board = []
    for row in range(rows):
        row = [0] * columns
        board.append(row)
    return board

def print_board(board): 
    """
    Print the board in the correct order

    Input: 
        board: the board to print

    Output:
        None
    """
    for row in reversed(board): 
        print(row)

def check_valid_move(board, column):
    """
    Check whether or not a move is valied

    Input: 
        board: the board to check
        column: the column to check

    Output:
        True if the move is allowed, False otherwise
    """
    return board[-1][column] == 0

def drop_piece(board, column, player): 
    """
    Drop a piece in a chosen column

    Input: 
        board: the board to drop the piece on
        column: the column to drop the piece in
        player: the player to drop the piece for

    Output:
        board: the board with the piece dropped or None if the move is invalid
    """
    board_copy = copy.deepcopy(board)
    if check_valid_move(board_copy, column):
        for row in range(len(board_copy)): 
            if board_copy[row][column] == 0: 
                board_copy[row][column] = player
                return board_copy
    else: 
        return None

def check_full_board(board): 
    """
    Check if the board is full

    Input: 
        board: the board to check_full_board

    Output:
        True if the board is full, False otherwise
    """
    for row in board:
        for column in row: 
            if column == 0: 
                return False
    return True

def check_win(board): 
    """
    Check if a player has won the game

    Input: 
        board: the board to check

    Output:
        1 if player 1 won
        2 if player 2 won
        None if no one has won
    """
    # Check horizontal
    for row in board: 
        for column in range(len(row) - 3): 
            if row[column] != 0 and row[column] == row[column + 1] == row[column + 2] == row[column + 3]: 
                return row[column]  
    
    # Check vertical
    for column in range(len(board[0])): 
        for row in range(len(board) - 3): 
            if board[row][column] != 0 and board[row][column] == board[row + 1][column] == board[row + 2][column] == board[row + 3][column]: 
                return board[row][column]  
    
    # Check diagonal
    for row in range(len(board) - 3): 
        for column in range(len(board[0]) - 3): 
            if board[row][column] != 0 and board[row][column] == board[row + 1][column + 1] == board[row + 2][column + 2] == board[row + 3][column + 3]: 
                return board[row][column]
    
    # Check diagonal
    for row in range(len(board) - 3): 
        for column in range(3, len(board[0])):  
            if board[row][column] != 0 and board[row][column] == board[row + 1][column - 1] == board[row + 2][column - 2] == board[row + 3][column - 3]: 
                return board[row][column] 
                
    return None 

def is_terminal_state(board): 
    """
    Check if the board is a terminal state

    Input: 
        board: the board to check

    Output:
        True if the board is a terminal state, False otherwise
    """
    if check_win(board) is not None or check_full_board(board): 
        return True
    return False

def format_board_for_llm(board):
    """
    Format the board in a way that makes them understandable for LLMs 

    Input:
        board: the board

    Output:
        formatted string of the board with column numbers
    """
    result = "  0 1 2 3 4 5 6  <- COLUMN NUMBERS\n"
    
    for i in range(len(board) - 1, -1, -1):
        row = board[i]
        row_string = ""
        for cell in row:
            row_string = row_string + str(cell) + " "
        result = result + row_string + "\n"
    
    return result


### Heuristic Functions

In [ ]:
def get_next_moves(board): 
    """
    Get the next moves for a player

    Input: 
        board: the board to get the next moves for  
        player: the player to get the next moves for

    Output:
        the next moves for the player
    """
    moves = []
    for column in range(len(board[0])): 
        if check_valid_move(board, column): 
            moves.append(column)
    return moves

def assign_board_score(board, player): 
    """
    Assign a score to a board

    Input: 
        board: the board to assign the score to
        player: the player for who we have to calculate the score

    Output:
        the score of the board
    """
    score = 0 

    # Check horizontal
    for row in range(len(board)): 
        for column in range(len(board[0]) - 3): 
            slice = [board[row][column], board[row][column+1], board[row][column+2], board[row][column+3]]
            positions = [(row, column), (row, column+1), (row, column+2), (row, column+3)]
            score += score_slice(slice, player, board, positions)

    # Check vertical
    for column in range(len(board[0])): 
        for row in range(len(board) - 3): 
            slice = [board[row][column], board[row+1][column], board[row+2][column], board[row+3][column]]
            positions = [(row, column), (row+1, column), (row+2, column), (row+3, column)]
            score += score_slice(slice, player, board, positions)

    # Check diagonal
    for row in range(len(board) - 3): 
        for column in range(len(board[0]) - 3): 
            slice = [board[row][column], board[row+1][column+1], board[row+2][column+2], board[row+3][column+3]]
            positions = [(row, column), (row+1, column+1), (row+2, column+2), (row+3, column+3)]
            score += score_slice(slice, player, board, positions)

    # Check diagonal
    for row in range(len(board) - 3): 
        for column in range(3, len(board[0])): 
            slice = [board[row][column], board[row+1][column-1], board[row+2][column-2], board[row+3][column-3]]   
            positions = [(row, column), (row+1, column-1), (row+2, column-2), (row+3, column-3)]
            score += score_slice(slice, player, board, positions)
    
    # bonus for center column positions
    center_column = len(board[0])//2
    for row in range(len(board)): 
        if board[row][center_column] == player: 
            score += 100
    
    return score
                
def score_slice(slice, player, board, positions): 
    """
    Give a score to a slice of the board

    input:
        slice: the slice of the board to score
        player: the player to score for
        board: the board to score for
        positions: the positions of the slice

    output:
        the score of the slice
    """
    score = 0
    opponent = 3 - player

    # Check playability of 0s
    unplayable_count = 0
    for i, value in enumerate(slice):
        if value == 0:
            row, column = positions[i]
            if row > 0 and board[row-1][column] == 0:
                unplayable_count += 1

    # assign positive score based on player position
    if slice.count(player) == 4: 
        score += 10000
    elif slice.count(player) == 3 and slice.count(0) == 1: 
        if unplayable_count == 1:
            score -= 300
        score += 400
    elif slice.count(player) == 2 and slice.count(0) == 2: 
        if unplayable_count == 1:
            score -= 50
        elif unplayable_count == 2:
            score -= 100
        score += 200
    else: 
        score += 0
    
    # assign negative score based on opponent position
    if slice.count(opponent) == 4 and slice.count(0) == 0: 
        score -= 10050
    elif slice.count(opponent) == 3 and slice.count(0) == 1: 
        if unplayable_count == 1:
            score += 300
        score -= 450
    elif slice.count(opponent) == 2 and slice.count(0) == 2: 
        if unplayable_count == 1:
            score += 50
        elif unplayable_count == 2:
            score += 100
        score -= 250
    else: 
        score -= 0

    return score


### Create MiniMax Player for Move Evaluation

In [ ]:
def minimax(board, player, depth, maximizing_player, alpha=-math.inf, beta=math.inf, memoization=None):
    """
    Minimax with alpha-beta pruning and memoization
    
    Input:
        board: board
        player: player to maximize for 
        depth: depth
        maximizing_player: True if maximizing, False if minimizing
        alpha: best value maximizing player
        beta: best value minimizing player
        memoization: dict to store position scores

    Output:
        best_score: the score for this position
    """
    opponent = 3 - player
    
    if memoization is not None:
        board_key = str(board)
        if board_key in memoization:
            cached_depth, cached_score = memoization[board_key]
            if cached_depth >= depth:
                return cached_score
    
    # Terminal state check
    if is_terminal_state(board):
        score = assign_board_score(board, player)
        if memoization is not None:
            memoization[str(board)] = (depth, score)
        return score
    
    if depth == 0:
        score = assign_board_score(board, player)
        if memoization is not None:
            memoization[str(board)] = (depth, score)
        return score
    
    valid_moves = get_next_moves(board)
    
    if maximizing_player:
        max_score = -math.inf
        for column in valid_moves:
            new_board = drop_piece(board, column, player)
            score = minimax(new_board, player, depth - 1, False, alpha, beta, memoization)
            max_score = max(max_score, score)
            alpha = max(alpha, score)
            if beta <= alpha:
                # Alpha-beta cutoff
                break
        if memoization is not None:
            memoization[str(board)] = (depth, max_score)
        return max_score
    
    else:  
        min_score = math.inf
        for column in valid_moves:
            new_board = drop_piece(board, column, opponent)
            score = minimax(new_board, player, depth - 1, True, alpha, beta, memoization)
            min_score = min(min_score, score)
            beta = min(beta, score)
            if beta <= alpha:
                # Alpha-beta cutoff
                break
        if memoization is not None:
            memoization[str(board)] = (depth, min_score)
        return min_score

def get_best_move_and_score(move_scores):
    """
    Find the column and score with the highest score
    
    Input:
        move_scores: list of tuples where each tuple is (column, score)
    
    Output:
        tuple of (best_column, best_score), or (None, -math.inf) if list is empty
    """
    if len(move_scores) == 0:
        return None, -math.inf
    
    best_score = -math.inf
    best_column = None
    
    for move_tuple in move_scores:
        column, score = move_tuple
        if score > best_score:
            best_score = score
            best_column = column
    
    return best_column, best_score

def sort_move_scores_by_score(move_scores):
    """
    Sort a list of (column, score) tuples by score in descending order
    
    Input:
        move_scores: list of tuples where each tuple is (column, score)
    
    Output:
        sorted list of tuples with highest scores first
    """
    sorted_move_scores = []
    remaining_scores = []
    
    for move_tuple in move_scores:
        remaining_scores.append(move_tuple)
    
    while len(remaining_scores) > 0:
        highest_score = -math.inf
        best_move_tuple = None
        best_index = 0
        
        for i in range(len(remaining_scores)):
            column, score = remaining_scores[i]
            if score > highest_score:
                highest_score = score
                best_move_tuple = remaining_scores[i]
                best_index = i
        
        sorted_move_scores.append(best_move_tuple)
        remaining_scores.pop(best_index)
    
    return sorted_move_scores

def order_moves(moves, previous_scores=None):
    """
    Order moves for better alpha-beta pruning efficiency
    
    Input:
        moves: list of valid column moves
        previous_scores: dict mapping column -> score from previous iteration
    
    Output:
        ordered list of moves
    """
    if previous_scores is None:
        previous_scores = {}
    
    move_priorities = []
    for col in moves:
        center_dist = abs(col - 3)
        center_score = 7 - center_dist
        prev_score = previous_scores.get(col, 0)
        
        # Multiply by 1000 to prioritize previous scores over center preference
        priority_score = prev_score * 1000 + center_score
        move_priorities.append((col, priority_score))
    
    sorted_moves = sort_move_scores_by_score(move_priorities)
    
    ordered_moves = []
    for move_tuple in sorted_moves:
        column, priority = move_tuple
        ordered_moves.append(column)
    
    return ordered_moves

def get_move_rankings(board, player, depth):
    """
    Get rankings for all valid moves using iterative deepening with memoization and move ordering
    
    Input:
        board: board 
        player: the player to move 
        depth: target depth for minimax (searches from depth 1 to depth + 4)
    
    Output:
        dict with rank 1=best and scores
    """
    valid_moves = get_next_moves(board)
    
    memoization = {}
    previous_scores = {}
    
    start_depth = 1
    end_depth = depth + 4
    
    move_scores = []
    best_move_stable = False
    
    for current_depth in range(start_depth, end_depth + 1):
        move_scores_current = []
        ordered_moves = order_moves(valid_moves, previous_scores)
        
        for column in ordered_moves:
            new_board = drop_piece(board, column, player)
            if is_terminal_state(new_board):
                score = assign_board_score(new_board, player)
                if score > 1000:
                    move_scores_current.append((column, score))
                    continue
            
            score = minimax(new_board, player, current_depth - 1, False, -math.inf, math.inf, memoization)
            move_scores_current.append((column, score))
            previous_scores[column] = score
        
        if move_scores:
            prev_best, _ = get_best_move_and_score(move_scores)
            new_best, _ = get_best_move_and_score(move_scores_current)
            
            if prev_best == new_best and not best_move_stable:
                best_move_stable = True
            elif prev_best != new_best:
                best_move_stable = False
        
        move_scores = move_scores_current
        
        if move_scores:
            _, best_score = get_best_move_and_score(move_scores)
            if best_score > 1000:
                break
    
    # Sort by score in descending order
    move_scores = sort_move_scores_by_score(move_scores)
    
    rankings = {}
    scores = {}
    current_rank = 1
    prev_score = None
    
    for i, (column, score) in enumerate(move_scores):
        scores[column] = score
        if prev_score is not None and score < prev_score:
            current_rank = i + 1
        rankings[column] = current_rank
        prev_score = score
    
    return {'rankings': rankings, 'scores': scores}


In [ ]:
def get_best_move_from_rankings(rankings):
    """
    Find the column with the best (lowest) rank from a rankings dictionary
    
    Input:
        rankings: dictionary mapping column -> rank (lower rank is better)
    
    Output:
        column number with the best rank, or None if dictionary is empty
    """
    if len(rankings) == 0:
        return None
    
    best_move = None
    best_rank = 999
    
    for col in rankings:
        rank = rankings[col]
        if rank < best_rank:
            best_rank = rank
            best_move = col
    
    return best_move

def evaluate_position(board, player, depth, get_moves):
    """
    Evaluate a position by comparing a chosen move against minimax rankings
    
    input:
        board: the current board state
        player: the player to move (1 or 2)
        depth: depth for minimax algorithm
        get_moves: function(board, player, valid_moves) that outputs chosen column (int) or dict with 'move' and 'rationale'
    
    output:
        dict with keys: 'chosen_move', 'ranking', 'rankings', 'correct', 'rationale' (if provided)
    """
    valid_moves = get_next_moves(board)
    
    ranking_data = get_move_rankings(board, player, depth)
    rankings = ranking_data['rankings']
    scores = ranking_data['scores']
    
    best_move = get_best_move_from_rankings(rankings)
    
    result = get_moves(board, player, valid_moves)
    
    if isinstance(result, dict):
        chosen_move = result['move']
        rationale = result.get('rationale', '')
    else:
        chosen_move = result
        rationale = None
    
    if chosen_move not in valid_moves:
        return {
            'chosen_move': chosen_move,
            'ranking': None,
            'rankings': rankings,
            'scores': scores,
            'correct': False,
            'error': 'Invalid move',
            'best_move': best_move,
            'rationale': rationale
        }
    
    ranking = rankings[chosen_move]
    
    return {
        'chosen_move': chosen_move,
        'ranking': ranking,
        'rankings': rankings,
        'scores': scores,
        'correct': ranking == 1,
        'best_move': best_move,
        'rationale': rationale
    }

def test_human(board, player, valid_moves):
    """
    Get move from human input
    
    input:
        board: current board
        player: the player to move (1 or 2)
        valid_moves: list of valid columns
    
    output:
        chosen column (int)
    """
    print_board(board)
    print()
    
    while True:
        try:
            column = int(input("col: "))
            return column
        except ValueError:
            pass


### Run Benchmarking

In [ ]:
def run_benchmark(get_moves, depth=6, save_to_file=None):
    """
    Core benchmarking function - runs any move getter through all position sets
    
    input:
        get_moves: function(board, player, valid_moves) -> column
        depth: minimax depth for evaluation
        save_to_file: filename to save results (e.g., 'human_results.json')
    
    output:
        dict with results for each position set
    """
    all_results = {}
    
    position_sets = [
        ("P1 Win in 1", p1_win_in_1, 1),
        ("P2 Win in 1 (Block)", p2_win_in_1_block, 1),
        ("P1 Win in 3", p1_win_in_3, 1),
        ("P2 Win in 3", p2_win_in_3, 1),
        ("P1 Win in 5", p1_win_in_5, 1),
        ("P2 Win in 5", p2_win_in_5, 1),
        ("Early Game", early_after_5_moves, 1),
        ("Late Game", late_after_20_moves, 1)
    ]
    
    for set_name, positions, player in position_sets:
        results = []
        for i, board in enumerate(positions):
            result = evaluate_position(board, player, depth, get_moves)
            
            clean_result = {
                'position_index': i,
                'category': set_name,
                'chosen_move': result['chosen_move'],
                'optimal_move': result['best_move'],
                'rank_distance': result['ranking'] - 1 if result['ranking'] is not None else None,
                'raw_rankings': result['rankings'],
                'raw_scores': result['scores']
            }
            if result.get('error'):
                clean_result['error'] = result['error']
            if result.get('rationale'):
                clean_result['rationale'] = result['rationale']
            
            results.append(clean_result)
        
        correct = 0
        for result in results:
            if result.get('rank_distance') == 0:
                correct += 1
        
        total = len(results)
        if total > 0:
            accuracy = correct / total
        else:
            accuracy = 0
        
        all_results[set_name] = {
            'correct': correct,
            'total': total,
            'accuracy': accuracy,
            'positions': results
        }
        print(f"Completed {set_name}: {correct}/{total} correct ({accuracy:.2%})")
    
    total_correct = 0
    total_positions = 0
    for set_name in all_results:
        total_correct += all_results[set_name]['correct']
        total_positions += all_results[set_name]['total']
    
    if total_positions > 0:
        overall_accuracy = total_correct / total_positions
    else:
        overall_accuracy = 0
    
    summary = {
        'total_correct': total_correct,
        'total_positions': total_positions,
        'overall_accuracy': overall_accuracy,
        'category_results': all_results
    }
    
    if save_to_file:
        with open(save_to_file, 'w') as f:
            json.dump(summary, f, indent=2)
    
    return summary

## LLM + Human Competetition 

In [ ]:
api_count = {'openai': 0, 'anthropic': 0, 'deepseek': 0, 'openai_failures': 0, 'anthropic_failures': 0, 'deepseek_failures': 0}

def get_openai_move(board, player, valid_moves, model="gpt-4o"):
    """
    Get move from OpenAI API
    
    input:
        board: current board state
        player: the player to move (1 or 2)
        valid_moves: list of valid column indices
        model: OpenAI model to use
    
    output:
        dict with 'move' (int) and 'rationale' (str)
    """
    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
    
    prompt = f"""You are playing Connect 4. You are Player 1 (marked as '1'), playing against Player 2 (marked as '2').

CRITICAL: Column numbering is 0-6 from LEFT TO RIGHT.

BOARD STATE (top to bottom, 0s are empty):
{format_board_for_llm(board)}

VALID MOVES: {valid_moves}

YOUR TASK: Select the single best move that either:
1. Wins the game immediately (gets 4 in a row)
2. Blocks opponent from winning on their next turn
3. Creates multiple winning threats
4. Maximizes your winning chances

Think strategically, then respond with your move choice and concise reasoning.

RESPONSE FORMAT: JSON with 'move' (integer 0-6) and 'rationale' (one sentence explaining your choice)."""

    while True:
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": "You are an expert Connect 4 player. Always respond with valid JSON containing 'move' (integer 0-6) and 'rationale' (string one sentence only)."},
                    {"role": "user", "content": prompt}
                ],
                max_tokens=200,
                temperature=0.1,
                response_format={"type": "json_object"}
            )
            
            content = response.choices[0].message.content.strip()
            result = json.loads(content)
            move = result['move']
            rationale = result['rationale']
            
            if move in valid_moves:
                api_count['openai'] += 1
                
                return {'move': move, 'rationale': rationale}
        except Exception as e:
            api_count['openai_failures'] += 1
            if api_count['openai_failures'] > 3:
                print(f"OpenAI failed >3 times. Exception: {e}")


def get_anthropic_move(board, player, valid_moves, model="claude-sonnet-4-5-20250929"):
    """
    Get move from Anthropic API
    
    input:
        board: current board state
        player: the player to move (1 or 2)
        valid_moves: list of valid column indices
        model: Anthropic model to use
    
    output:
        dict with 'move' (int) and 'rationale' (str)
    """
    client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
    
    prompt = f"""You are playing Connect 4. You are Player 1 (marked as '1'), playing against Player 2 (marked as '2').

CRITICAL: Column numbering is 0-6 from LEFT TO RIGHT.

BOARD STATE (top to bottom, 0s are empty):
{format_board_for_llm(board)}

VALID MOVES: {valid_moves}

YOUR TASK: Select the single best move that either:
1. Wins the game immediately (gets 4 in a row)
2. Blocks opponent from winning on their next turn
3. Creates multiple winning threats
4. Maximizes your winning chances

Think strategically, then respond with your move choice and concise reasoning.

RESPONSE FORMAT: JSON with 'move' (integer 0-6) and 'rationale' (one sentence explaining your choice)."""

    while True:
        try:
            message = client.beta.messages.create(
                model=model,
                max_tokens=200,
                temperature=0.1,
                betas=["structured-outputs-2025-11-13"],
                system="You are an expert Connect 4 player.",
                messages=[
                    {"role": "user", "content": prompt}
                ],
                output_format={
                    "type": "json_schema",
                    "schema": {
                        "type": "object",
                        "properties": {
                            "move": {
                                "type": "integer",
                                "description": "The column number (0-6) to drop the piece"
                            },
                            "rationale": {
                                "type": "string",
                                "description": "A one-sentence explanation why you chose the move"
                            }
                        },
                        "required": ["move", "rationale"],
                        "additionalProperties": False
                    }
                }
            )
            
            content = message.content[0].text.strip()
            result = json.loads(content)
            move = result['move']
            rationale = result['rationale']
            
            if move in valid_moves:
                api_count['anthropic'] += 1
                
                return {'move': move, 'rationale': rationale}
        except Exception as e:
            api_count['anthropic_failures'] += 1
            if api_count['anthropic_failures'] > 3:
                print(f"Anthropic failed >3 times. Exception: {e}")


def get_deepseek_move(board, player, valid_moves, model="deepseek-chat"):
    """
    Get move from DeepSeek API
    
    input:
        board: current board state
        player: the player to move (1 or 2)
        valid_moves: list of valid column indices
        model: DeepSeek model to use
    
    output:
        dict with 'move' (int) and 'rationale' (str)
    """
    client = OpenAI(api_key=os.getenv("DEEPSEEK_API_KEY"), base_url="https://api.deepseek.com")
    
    prompt = f"""You are playing Connect 4. You are Player 1 (marked as '1'), playing against Player 2 (marked as '2').

CRITICAL: Column numbering is 0-6 from LEFT TO RIGHT.

BOARD STATE (top to bottom, 0s are empty):
{format_board_for_llm(board)}

VALID MOVES: {valid_moves}

YOUR TASK: Select the single best move that either:
1. Wins the game immediately (gets 4 in a row)
2. Blocks opponent from winning on their next turn
3. Creates multiple winning threats
4. Maximizes your winning chances

Think strategically, then respond with your move choice and concise reasoning.

RESPONSE FORMAT: JSON with 'move' (integer 0-6) and 'rationale' (one sentence explaining your choice)."""

    while True:
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": "You are an expert Connect 4 player. Always respond with valid JSON containing 'move' (integer 0-6) and 'rationale' (string one sentence only)."},
                    {"role": "user", "content": prompt}
                ],
                max_tokens=200,
                temperature=0.1,
                response_format={"type": "json_object"}
            )
            
            content = response.choices[0].message.content.strip()
            result = json.loads(content)
            move = result['move']
            rationale = result['rationale']
            
            if move in valid_moves:
                api_count['deepseek'] += 1
                
                return {'move': move, 'rationale': rationale}
        except Exception as e:
            api_count['deepseek_failures'] += 1
            if api_count['deepseek_failures'] > 3:
                print(f"DeepSeek failed >3 times. Exception: {e}")


## Using the Code

In [ ]:
# For the assignment I stored the output in the json files in this directory. 
# If you wish to run the code yourself, uncomment the following lines and place your API keys in a .env file
# If you wish to test it on yourself only, uncomment the last line and press run all (make sure the data paths imported at the start of the file are correct)

#run_benchmark(get_deepseek_move, depth=6, save_to_file='deepseek_results.json')
#run_benchmark(get_openai_move, depth=6, save_to_file='openai_results2.json')
#run_benchmark(get_anthropic_move, depth=6, save_to_file='anthropic_results.json')
#run_benchmark(test_human, depth=6, save_to_file='human_results.json')